# RSA — Reviewer Experiments

This notebook runs the experiments requested by the critical review:

**Quality / novelty baselines**
- zero-shot MiniLM concept vectors
- zero-shot prompt-difference vectors
- no rotation + 4-bit sparse LUT
- PCA rotation + 4-bit sparse LUT
- random rotation RSA
- PQ64 (64 B/item) + compiled linear LUT head
- FP32 linear semantic proxy
- 384→64→8 MLP ceiling
- oracle

**Systems benchmark**
- 384-D FP32 fused linear heads
- PQ64 fused LUT heads
- packed 4-bit RSA item-major
- packed 4-bit RSA coordinate-major
- fused multi-predicate RSA plan
- i16 quantized LUTs

The benchmark also stratifies the earlier composition-depth result by conjunction prevalence.


In [ ]:
#@title 1) Settings
FULL_QUALITY = False #@param {type:"boolean"}
RUST_THROUGHPUT_ITEMS = 500000 #@param {type:"integer"}
RUST_CAPACITY_ITEMS = 5000000 #@param {type:"integer"}
RUST_REPEATS = 5 #@param {type:"integer"}

print("FULL_QUALITY =", FULL_QUALITY)


In [ ]:
#@title 2) Clone repo and install Python + PQ dependencies
import os, pathlib, shutil, subprocess
ROOT=pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['pip','install','-q','-e','.','faiss-cpu'],check=True)
print("repo:", subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())


In [ ]:
#@title 3) Run quality baselines
import subprocess, time, os
os.chdir('/content/ras')
cfg = 'configs/reviewer_baselines.yaml' if FULL_QUALITY else 'configs/reviewer_baselines_smoke.yaml'
print("config:", cfg)
t0=time.time()
subprocess.run(['python','-m','experiments.reviewer_baselines','--config',cfg],check=True)
print(f"quality suite finished in {(time.time()-t0)/60:.1f} min")


In [ ]:
#@title 4) Show quality results
from pathlib import Path
import json, pandas as pd
root=Path('/content/ras/results')
runs=sorted([p for p in root.iterdir() if p.is_dir() and '_reviewer_' in p.name],key=lambda p:p.stat().st_mtime)
run=runs[-1]
print("run:",run)
headline=json.loads((run/'headline.json').read_text())
print(json.dumps(headline,indent=2))
summary=pd.read_csv(run/'summary.csv')
pred=pd.read_csv(run/'predicate_metrics.csv')
deltas=pd.read_csv(run/'paired_deltas.csv')
display(pred.groupby('method')[['f1','ap']].mean().sort_values('f1',ascending=False))
display(summary[(summary.metric=='recall') & (summary.retention.isin([0.4,0.2,0.1]))].pivot(index='method',columns='retention',values='mean').sort_values(0.2,ascending=False))


In [ ]:
#@title 5) Check composition depth after controlling for prevalence
import pandas as pd
depth=pd.read_csv(run/'depth_prevalence_stratified.csv')
display(depth)
print("If the raw depth trend was mainly prevalence-driven, the within-bin means should no longer rise monotonically with n_latents.")


In [ ]:
#@title 6) Install Rust if needed
import shutil, subprocess, os
if shutil.which('cargo') is None:
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','cargo','rustc'],check=True)
print(subprocess.check_output(['rustc','--version']).decode().strip())
print(subprocess.check_output(['cargo','--version']).decode().strip())


In [ ]:
#@title 7) Compile/test and run Rust systems benchmark
import subprocess, os, time
os.chdir('/content/ras')
manifest='rust/semantic_engine/Cargo.toml'
subprocess.run(['cargo','test','--release','--manifest-path',manifest],check=True)
out='/content/rsa_systems_results.csv'
t0=time.time()
subprocess.run([
    'cargo','run','--release','--manifest-path',manifest,'--',
    '--throughput-items',str(RUST_THROUGHPUT_ITEMS),
    '--capacity-items',str(RUST_CAPACITY_ITEMS),
    '--repeats',str(RUST_REPEATS),
    '--out',out
],check=True)
print(f"systems benchmark finished in {(time.time()-t0)/60:.1f} min")


In [ ]:
#@title 8) Show systems results
import pandas as pd
sysdf=pd.read_csv('/content/rsa_systems_results.csv')
display(sysdf.sort_values(['predicates','candidates','million_candidates_per_s'],ascending=[True,True,False]))
print('\n100k-candidate / 1-predicate comparison')
display(sysdf[(sysdf.candidates==100000)&(sysdf.predicates==1)].sort_values('million_candidates_per_s',ascending=False))


In [ ]:
#@title 9) Build the quality × memory × throughput table
import pandas as pd, numpy as np
quality=summary[(summary.metric=='recall')&(summary.retention==0.2)][['method','mean']].rename(columns={'mean':'recall_at_20pct'})
meta=pd.read_csv(run/'method_meta.csv')
q=quality.merge(meta,on='method',how='left')
s=sysdf[(sysdf.candidates==100000)&(sysdf.predicates==1)].copy()
best_rsa=s[s.representation=='rsa4_f32'].sort_values('million_candidates_per_s',ascending=False).head(1)
rows=[]
for method, repr_name in [('linear_fp32','fp32_linear'),('pq64_linear_lut','pq64_lut_head')]:
    z=s[s.representation==repr_name].sort_values('million_candidates_per_s',ascending=False).head(1)
    if len(z): rows.append((method,float(z.iloc[0].million_candidates_per_s),z.iloc[0]['layout']))
if len(best_rsa): rows.append(('lut_random',float(best_rsa.iloc[0].million_candidates_per_s),best_rsa.iloc[0]['layout']))
speed=pd.DataFrame(rows,columns=['method','million_candidates_per_s','systems_layout'])
pareto=q.merge(speed,on='method',how='left').sort_values('recall_at_20pct',ascending=False)
display(pareto)


In [ ]:
#@title 10) Display figures and zip results
from IPython.display import display, Image
import shutil
for p in sorted((run/'figures').glob('*.png')):
    print(p.name); display(Image(filename=str(p)))
bundle=Path('/content/rsa_reviewer_bundle')
bundle.mkdir(exist_ok=True)
shutil.copytree(run,bundle/'quality',dirs_exist_ok=True)
shutil.copy('/content/rsa_systems_results.csv',bundle/'systems_results.csv')
zip_path=shutil.make_archive('/content/rsa_reviewer_experiments','zip',root_dir=str(bundle))
print(zip_path)
